# Check sampling — width-distilled EDM

Teacher 420 vs student 256/128, **ancestral** `EquivariantDiffusion` (ε-pred, production sampler).
Gate **100 steps first**; 8-step is a bonus, not the training target.

Point `CKPT` at `checkpoints_edm/best_256_edm.pt` (or `latest_…` / `best_128_edm.pt`).
`USE_EMA=False` by default — non-EMA won the PD runs; flip it to compare.

In [ ]:
"""Teacher 420 EDM vs width-distilled student, ancestral sampling at 100 and 8 steps."""
import math
import time
from pathlib import Path

import py3Dmol
import torch
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")

from src.mlconfgen.egnn import EGNNDynamics
from src.mlconfgen.equivariant_diffusion import EquivariantDiffusion, PredefinedNoiseSchedule
from src.mlconfgen.utils import (
    ATOM_DECODER, CONTEXT_NORMS, MAX_N_NODES,
    align_mol_to_principal_frame, prepare_edm_input, samples_to_rdkit_mol,
)

device = "cpu"
N_SHOW, SEED = 4, 42
STEPS = (100, 20)            # 100 is the gate; 8 is extra
MOL = "./assets/demo_files/ceyyag.mol"          # 17 atoms; try 38_atoms_example.mol for the hard case
EDM = Path("./edm_moi_chembl_15_39.pt")
CKPT = Path("./best_256_edm.pt")  # or latest_256_edm.pt / best_128_edm.pt
USE_EMA = False             # True -> student_ema weights
NOISE_PRECISION = 1e-5

torch.manual_seed(SEED)


def show(mols):
    blocks = [Chem.MolToXYZBlock(m) for m in mols if m is not None]
    if not blocks:
        print("nothing to show")
        return
    cols = min(4, len(blocks))
    rows = math.ceil(len(blocks) / cols)
    v = py3Dmol.view(viewergrid=(rows, cols), width=250 * cols, height=250 * rows)
    for i, b in enumerate(blocks):
        r, c = divmod(i, cols)
        v.addModel(b, "xyz", viewer=(r, c))
        v.setStyle({"stick": {"radius": 0.15}, "sphere": {"scale": 0.25}}, viewer=(r, c))
        v.zoomTo(viewer=(r, c))
    v.show()


def make_edm(hidden_nf, ckpt=None, initial_steps=1000, steps=100):
    dyn = EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=hidden_nf, device=device)
    mdl = EquivariantDiffusion(dyn, in_node_nf=8, timesteps=initial_steps, noise_precision=NOISE_PRECISION)
    if ckpt:
        mdl.load_state_dict(ckpt)
    set_steps(mdl, steps)
    return mdl.to(device).eval()


def set_steps(mdl, n):
    mdl.gamma = PredefinedNoiseSchedule(timesteps=n, precision=NOISE_PRECISION)
    mdl.time_steps = torch.flip(torch.arange(0, n, device=device), dims=[0])
    mdl.T = n


def report(x, nm, tag):
    m = nm[..., 0].bool()
    xs = x[m]
    finite = torch.isfinite(xs).all().item()
    std = xs.std().item() if finite else float("nan")
    n_bad = int((~torch.isfinite(x).reshape(x.shape[0], -1).all(1)).sum())
    print(f"{tag}  coord std={std:.3f}  finite={finite}  nonfinite mols={n_bad}/{x.shape[0]}")


@torch.inference_mode()
def sample(model, nm, em, ctx, n_steps, tag):
    set_steps(model, n_steps)
    t0 = time.perf_counter()
    x, h = model(nm, em, ctx)
    dt = time.perf_counter() - t0
    print(f"{tag} NFE={n_steps}: {dt * 1000 / nm.shape[0]:.0f} ms/mol")
    report(x, nm, tag)
    return x, h


def resolve_ckpt(path: Path) -> Path:
    if path.exists():
        return path
    hits = sorted(Path("./checkpoints_edm").glob("best_*_edm.pt"), key=lambda p: p.stat().st_mtime)
    hits += sorted(Path("./checkpoints_edm").glob("latest_*_edm.pt"), key=lambda p: p.stat().st_mtime)
    if not hits:
        raise FileNotFoundError(f"{path} missing and no checkpoints_edm/*_edm.pt")
    print(f"{path} missing — using newest {hits[-1].name}")
    return hits[-1]


# --------------------- load ---------------------
edm_ckpt = torch.load(EDM, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}

teacher = make_edm(420, edm_ckpt["state_dict"])
for p in teacher.parameters():
    p.requires_grad_(False)

path = resolve_ckpt(CKPT)
sc = torch.load(path, map_location=device, weights_only=False)
width = int(sc.get("hidden_nf", 256))
key = "student_ema" if USE_EMA and "student_ema" in sc else ("student" if "student" in sc else "state_dict")
print(
    f"student {path.name}  hidden={width}  weights={key}  "
    f"epoch={sc.get('epoch')}  avg_loss={sc.get('avg_loss')}"
)
student = make_edm(width, sc[key], 100)

for p in student.parameters():
    p.requires_grad_(False)

ref = Chem.RemoveAllHs(Chem.MolFromMolFile(MOL))
ctx3, *_ = align_mol_to_principal_frame(ref)
n = ref.GetNumAtoms()
nm, em, context = prepare_edm_input(N_SHOW, ctx3.to(device), norms, n, n, device, pad_to=MAX_N_NODES)
print(f"ref={Path(MOL).name}  n_atoms={n}  batch={N_SHOW}  device={device}")

for n_steps in STEPS:
    print(f"\n=== ancestral {n_steps} steps ===")
    torch.manual_seed(SEED)
    xt, ht = sample(teacher, nm, em, context, n_steps, "teacher420")
    show(samples_to_rdkit_mol(xt.cpu(), ht.cpu(), nm.cpu(), ATOM_DECODER))
    torch.manual_seed(SEED)
    xs, hs = sample(student, nm, em, context, n_steps, f"student{width}")
    show(samples_to_rdkit_mol(xs.cpu(), hs.cpu(), nm.cpu(), ATOM_DECODER))

student best_256_edm.pt  hidden=256  weights=student  epoch=6  avg_loss=0.05018882436859899
ref=ceyyag.mol  n_atoms=17  batch=4  device=cpu

=== ancestral 100 steps ===
